# Agent CRUD — `@azure/ai-projects`

This notebook demonstrates basic **agent** operations from the Azure Agents service: create versions, retrieve by name/version, list versions, and delete.

It mirrors the [`agentCurd.ts`](./agentCurd.ts) sample and runs the **locally built** `@azure/ai-projects` from this repo.

## Prerequisites

1. **Build the package first** so `dist/` is current: `cd sdk/ai/ai-projects && pnpm build`
2. **tslab kernel** installed and registered (`npm install -g tslab` then `tslab install`); select the **TypeScript** (tslab) kernel.
3. **Launch VS Code / Jupyter from `sdk/ai/ai-projects/`** so Node resolves the local `@azure/ai-projects`.
4. **`az login`** completed so `DefaultAzureCredential` can authenticate.
5. **Environment variables**: `FOUNDRY_PROJECT_ENDPOINT`, `FOUNDRY_MODEL_NAME`.

Run the cells in order (top to bottom); state is shared across cells.

In [6]:
// Imports and configuration
import { DefaultAzureCredential } from "@azure/identity";
import { AIProjectClient } from "@azure/ai-projects";

const projectEndpoint = process.env["FOUNDRY_PROJECT_ENDPOINT"] ?? "<project endpoint>";
const deploymentName = process.env["FOUNDRY_MODEL_NAME"] ?? "<model deployment name>";

console.log(`Model: ${deploymentName}`);

Model: gpt-5.2


In [7]:
// Create the AI Project client
const credential = new DefaultAzureCredential();
const project = new AIProjectClient(projectEndpoint, credential);

In [8]:
// Create two agent versions
const agent1 = await project.agents.createVersion("bg-crud-agent4", {
  kind: "prompt",
  model: deploymentName,
});
console.log("Created agent id:", agent1.id, "version:", agent1.version, "name:", agent1.name);

const agent2 = await project.agents.createVersion("bg-crud-agent2", {
  kind: "prompt",
  model: deploymentName,
});
console.log("Created agent id:", agent2.id, "version:", agent2.version, "name:", agent2.name);

Created agent id: bg-crud-agent4:1 version: 1 name: bg-crud-agent4
Created agent id: bg-crud-agent2:1 version: 1 name: bg-crud-agent2


In [9]:
// Retrieve agent by name and version
const retrievedAgent1 = await project.agents.getVersion(agent1.name, agent1.version);
console.log(
  "Retrieved agent id:",
  retrievedAgent1.id,
  "version:",
  retrievedAgent1.version,
  "name:",
  retrievedAgent1.name,
);

// Retrieve agent by name (latest version)
const latestAgent1 = await project.agents.get(agent1.name);
console.log(
  "Retrieved latest agent id:",
  latestAgent1.id,
  "name:",
  latestAgent1.versions.latest.name,
);

Retrieved agent id: bg-crud-agent4:1 version: 1 name: bg-crud-agent4
Retrieved latest agent id: bg-crud-agent4 name: bg-crud-agent4


In [10]:
// List all versions of agent1
// NOTE: tslab's transpiler rejects top-level `for await` (SyntaxError:
// Unexpected reserved word), even when wrapped in an async IIFE. Iterate the
// PagedAsyncIterableIterator manually with .next() instead.
console.log("List all agents:");
const iterator = project.agents.listVersions(agent1.name)[Symbol.asyncIterator]();
let next = await iterator.next();
while (!next.done) {
  const item = next.value;
  console.log("Agent id:", item.id, "name:", item.name, "version:", item.version);
  next = await iterator.next();
}

List all agents:
Agent id: bg-crud-agent4:1 name: bg-crud-agent4 version: 1


In [11]:
// Delete both agents
const result1 = await project.agents.deleteVersion(agent1.name, agent1.version);
console.log(
  "Deleted agent name:",
  result1.name,
  "version:",
  agent1.version,
  "result:",
  result1.deleted,
);

const result2 = await project.agents.deleteVersion(agent2.name, agent2.version);
console.log(
  "Deleted agent name:",
  result2.name,
  "version:",
  agent2.version,
  "result:",
  result2.deleted,
);

Deleted agent name: bg-crud-agent4 version: 1 result: true
Deleted agent name: bg-crud-agent2 version: 1 result: true
